# Ingest — Copilot Studio

Reads the three Power Platform admin center exports from `Files/landing/studio/` into
`studio_tenant_daily`, `studio_agent` and `studio_user`.

**Optional, off by default:** it can also read the files a Power Automate flow saves each day from
the Power Platform licensing API (`StudioApiAgentDaily_*.csv`, `StudioApiUserDaily_*.csv` and
`StudioApiEntitlement_*.csv`). A flow signed in as an admin is the only way to call that API on a
schedule: its licensing permissions are delegated only, so a notebook can't. Set
`READ_API_FILES = True` once a flow saves the files here. They give credits by day per agent and per
user, kept in `studio_agent_daily` and `studio_user_daily`, and rolled up into the other three tables
with `source_file = 'ppac-api'`. When only the per-user file has a day (the agent file is missing or
empty), `studio_tenant_daily` is summed from `studio_user_daily` instead, with
`source_file = 'ppac-api-users'`, so the totals are never blank while user-level credits exist.
An export always wins: API rows are dropped wherever an export covers the same day and environment,
or the same month for agents and users. With the API files on, the exports become optional: they
add the user's email as PPAC shows it, the M365 Copilot licence flag and the exact prepaid split.

## The awkward bit: two of these have no date

`StudioTenantDaily.csv` carries `Usage Date`, so each row lands on its own day.

`StudioPerAgent.csv` and `StudioPerUser.csv` do **not**. They are period aggregates for whatever
window the portal had selected — PPAC shows month-to-date plus the last two full months. Appending
them blindly would double-count on every re-run; overwriting them would throw away last month.

So each export counts as the month (UTC) its file was saved into `Files/landing/studio/`, kept as
`snapshot_month` and part of the key. A newer export in the same month updates that month's rows;
one saved next month adds a new month. An export left in the folder keeps re-stating its own
month, so it never becomes a later month's snapshot. **The result is per-agent history from a
source that has none** — which is most of the point of running this in Fabric at all.

To load an earlier month's export (last month's, say, exported on the 1st), put the file in
`Files/landing/studio/backfill/<YYYY-MM>/`, the folder for the month it covers: it counts as that
month on every run. SharePoint and installer uploads reach `Files/landing/studio/` when the upload
router next runs, as the pipeline's first step, so a file sent that way counts as the month of that
run. One uploaded on a month's last day may not be routed until the next month, so put that one in
its backfill folder.

One consequence worth understanding: the snapshot is a *cumulative month-to-date* figure, not a
monthly increment. Comparing two snapshots within the same month shows growth to date, not new
consumption. Consumption Central treats these as period totals for exactly this reason and never plots them
on a time axis.

In [ ]:
LANDING = "Files/landing/studio"

# The per-agent and per-user exports have no date, so each counts as the month (UTC) its file was
# saved into LANDING. To load an earlier month's export, put it in LANDING/backfill/<YYYY-MM>/
# instead: it counts as that folder's month on every run.

# The real PPAC export names are tried first; the loose pattern catches a file
# someone has renamed. The first pattern that matches a file with the export's
# key columns wins, so *Tenant*.csv never reads the per-agent or per-user export.
PAT_TENANT = ("EntitlementConsumptionTenantDetailsReport_MCSMessages*.csv",
              "*Tenant*.csv")
PAT_AGENT = ("EntitlementConsumptionTenantPerAgentDetailsReport_MCSMessages*.csv",
             "*PerAgent*.csv", "*Agent*.csv")
PAT_USER = ("EntitlementConsumptionTenantPerUserDetailsReport_MCSMessages*.csv",
            "*PerUser*.csv", "*User*.csv")

# Off by default, so only the exports above are read. Set True once a Power Automate flow saves the
# licensing API files (StudioApi*.csv) into LANDING.
READ_API_FILES = False

# Saved by that flow. The export patterns above skip them.
PAT_API_AGENT = ("StudioApiAgentDaily*.csv",)
PAT_API_ENTITLEMENT = ("StudioApiEntitlement*.csv",)
PAT_API_USER = ("StudioApiUserDaily*.csv",)

TBL_TENANT = "studio_tenant_daily"
TBL_AGENT = "studio_agent"
TBL_USER = "studio_user"
TBL_AGENT_DAILY = "studio_agent_daily"
TBL_USER_DAILY = "studio_user_daily"

API_SOURCE = "ppac-api"   # source_file on every row that came from the licensing API
# source_file on studio_tenant_daily rows summed from studio_user_daily, for days nothing else covers
API_USER_SOURCE = "ppac-api-users"
API_SOURCES = (API_SOURCE, API_USER_SOURCE)
API_BILLING_PLAN = "00000000-0000-0000-0000-000000000000"

# Only for looking up the UPN of each user the licensing API gives as an Entra object ID.
# Needs Microsoft Graph User.Read.All (application), as Ingest_Org_Graph does. Without it
# the run carries on and those users show by ID.
TENANT_ID = "<your-tenant-guid>"
CLIENT_ID = "<your-app-reg-client-id>"
CLIENT_SECRET = "<your-client-secret-value>"   # prod: notebookutils.credentials.getSecret("<kv-uri>", "<name>")
RESOLVE_UPNS = True


In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import date, datetime, timezone
import re


def norm(name):
    return re.sub(r"[ _\-/]", "", name).lower()


def pick(df, *aliases):
    """Actual column name matching any alias, ignoring case and separators.

    The PPAC headers contain spaces and a slash ('AI Feature/Billable Feature'),
    and are the sort of thing that gets tidied up between releases.
    """
    lookup = {norm(c): c for c in df.columns}
    for a in aliases:
        if norm(a) in lookup:
            return lookup[norm(a)]
    return None


MONTH_FOLDER = re.compile(r"\d{4}-(0[1-9]|1[0-2])")


def backfill():
    """(path under LANDING, file name, month, modifyTime) for each file in LANDING/backfill/<YYYY-MM>/.

    Stops before anything is written if backfill/ holds a file outside a month folder, a folder whose
    name isn't a month, or a month after this one.
    """
    found, problems = [], []
    for e in notebookutils.fs.ls(f"{LANDING}/backfill"):
        folder = e.name.rstrip("/")
        path = f"{LANDING}/backfill/{folder}"
        if not e.isDir:
            problems.append(f"{path}: a file outside a month folder")
        elif not MONTH_FOLDER.fullmatch(folder):
            problems.append(f"{path}: the folder name isn't a month (YYYY-MM)")
        elif date.fromisoformat(f"{folder}-01") > RUN_MONTH:
            problems.append(f"{path}: a month after this one ({RUN_MONTH:%Y-%m})")
        else:
            month = date.fromisoformat(f"{folder}-01")
            for f in notebookutils.fs.ls(path):
                if f.isDir:
                    print(f"  ! {path}/{f.name.rstrip('/')}: a folder inside a month folder isn't read")
                else:
                    found.append((f"backfill/{folder}/{f.name}", f.name, month, getattr(f, "modifyTime", None)))
    if problems:
        raise ValueError("Nothing was written. Each export in backfill/ goes in a folder named for its month, "
                         "YYYY-MM, no later than this one. Move or remove these, then run again:\n  "
                         + "\n  ".join(problems))
    return found


def read(*patterns, api=False, need=()):
    """Read every CSV matching any pattern, unioned, with file lineage.

    The flow's files (StudioApi*) are read only with api=True, so a loose
    export pattern such as *Agent*.csv never picks them up. They're read from
    LANDING itself; the exports from LANDING and its backfill/<YYYY-MM>/ folders.

    need lists the columns an export must have, each as its aliases. A file
    without them is skipped before the patterns are weighed, so the loose
    *Tenant*.csv never reads the per-agent or per-user export.

    Each export row gets _month, the month the export counts as: its backfill
    folder's month, else the month (UTC) its file was saved into LANDING. It
    also gets _order, higher for a newer file, so the newest file wins a key.

    multiLine and escape matter more than they look: Knowledge Sources,
    Scenario Name and Agent Name are free text, and one comma or line break
    inside a value silently shifts every column after it for that row.

    Union by name rather than reading the glob in a single call, so one export
    having a column the others lack does not fail the load - which happens when
    exports are taken weeks apart across a product update.
    """
    if api and not READ_API_FILES:
        print("  licensing API files are off (READ_API_FILES = False)")
        return None
    try:
        entries = notebookutils.fs.ls(LANDING)
    except Exception as e:
        print(f"  cannot list {LANDING}: {e}")
        return None
    files = [(e.name, e.name, None, getattr(e, "modifyTime", None)) for e in entries if not e.isDir]
    if not api and any(e.isDir and e.name.rstrip("/") == "backfill" for e in entries):
        files += backfill()
    files = [f for f in files if f[1].lower().endswith(".csv") and f[1].lower().startswith("studioapi") == api]

    found = []
    for p in patterns:
        rx = re.compile("^" + re.escape(p).replace("\\*", ".*") + "$", re.IGNORECASE)
        for path, name, month, modified in files:
            if not rx.match(name):
                continue
            df = (spark.read
                  .option("header", True)
                  .option("inferSchema", False)
                  .option("multiLine", True)
                  .option("escape", '"')
                  .option("encoding", "UTF-8")
                  .csv(f"{LANDING}/{path}"))
            missing = [aliases[0] for aliases in need if pick(df, *aliases) is None]
            if missing:
                print(f"  {path}: skipped, missing column(s): {', '.join(missing)}")
                continue
            found.append((path, name, month, modified, df))
        if found:
            break          # first pattern that matches a usable file wins

    if not found:
        print(f"  no file matched {patterns}")
        return None

    # Oldest first: the API files by name, as before; the exports by when they were saved, then by name.
    found.sort(key=lambda f: (f[1], f[0]) if api else (f[3] or 0, f[1], f[0]))
    frames = []
    for order, (path, name, month, modified, df) in enumerate(found):
        df = df.withColumn("source_file", F.lit(name))
        if not api:
            if month is None and modified:
                month = datetime.fromtimestamp(modified / 1000, timezone.utc).date().replace(day=1)
            elif month is None:
                month = RUN_MONTH
                print(f"  ! {path}: no modifyTime, so it counts as this month, {RUN_MONTH:%Y-%m}")
            df = df.withColumn("_month", F.lit(month)).withColumn("_order", F.lit(order))
        frames.append(df)

    out = frames[0]
    for df in frames[1:]:
        out = out.unionByName(df, allowMissingColumns=True)

    n = out.count()
    print(f"  {len(found)} file(s), {n:,} rows: {', '.join(f[0] for f in found)}")
    return out if n else None


def date_in(column, days):
    """SQL that's true where the column holds one of these dates. A list, not a range: a date the file
    doesn't have keeps its rows."""
    dates = ", ".join(f"DATE '{d.isoformat()}'" for d in sorted(set(days)))
    return f"{column} IN ({dates})"


def replace_rows(df, table, where, keep=None):
    """Writes df in place of the rows `where` selects, in one Delta commit, so a failed write leaves the
    table as it was. keep holds stored rows that `where` selects but that stay: they're written back.

    Delta fails the write if a row doesn't match `where`. A column the table doesn't have is left out,
    and a table column df lacks is an error, raised here before anything is written.
    """
    if not spark.catalog.tableExists(table):
        df.write.format("delta").saveAsTable(table)
        print(f"  {table}: created with {df.count():,} rows")
        return
    before = spark.table(table).count()
    rows = df.select(*[F.col(f.name).cast(f.dataType).alias(f.name) for f in spark.table(table).schema])
    if keep is not None:
        rows = rows.unionByName(keep)
    (rows.write.format("delta").mode("overwrite")
         .option("replaceWhere", where).saveAsTable(table))
    after = spark.table(table).count()
    print(f"  {table}: {before:,} -> {after:,}  ({after - before:+,})")


def same(columns, a="t", b="s"):
    """SQL that's true where rows a and b hold the same values in these columns, nulls included."""
    return " AND ".join(f"{a}.{c} <=> {b}.{c}" for c in columns)


def write_export(df, table, keys, covers):
    """Writes an export in one Delta commit, leaving what loading its files one at a time, oldest first,
    would leave.

    keys is the table's key. Where two files share a key, the newer file's row is written; rows that
    share a key within one file are all written. Stored rows in the export's days (or months) stay,
    unless the export has their key, or they're licensing API rows an export row covers: the same
    values in covers, the day and environment for the tenant export and the month for the others.
    Days and months the export doesn't have keep their rows.
    """
    rows = (df.alias("t").join(df.alias("s"), F.expr(f"{same(keys)} AND s._order > t._order"), "left_anti")
            .drop("_order"))
    values = [r[0] for r in rows.select(covers[0]).distinct().collect()]
    dated = [v for v in values if v is not None]
    parts = [date_in(covers[0], dated)] if dated else []
    if len(dated) < len(values):
        parts.append(f"{covers[0]} IS NULL")
    where = parts[0] if len(parts) == 1 else f"({' OR '.join(parts)})"
    keep = None
    if spark.catalog.tableExists(table):
        api = f"t.source_file IN ('{API_SOURCE}', '{API_USER_SOURCE}')"
        keep = (spark.table(table).where(where).alias("t")
                .join(rows.alias("s"), F.expr(same(keys)), "left_anti").alias("t")
                .join(rows.select(*covers).distinct().alias("s"), F.expr(f"{same(covers)} AND {api}"),
                      "left_anti"))
    replace_rows(rows, table, where, keep=keep)


def opt(df, *aliases):
    """The column if the file has it, else null."""
    c = pick(df, *aliases)
    return F.col(c) if c else F.lit(None)


def prepaid_share(prepaid, payg, entitled):
    """The tenant's month-to-date prepaid share: all prepaid while there's an entitlement and nothing used."""
    prepaid, payg, entitled = prepaid or 0.0, payg or 0.0, entitled or 0.0
    if prepaid + payg > 0:
        return prepaid / (prepaid + payg)
    return 1.0 if entitled > 0 else 0.0


_entitlement_cache = {}


def entitlement():
    """(prepaid share, {environment id: allocation}) from the latest entitlement snapshot, read once."""
    if "share" in _entitlement_cache:
        return _entitlement_cache["share"], _entitlement_cache["allocated"]
    ent_raw = read(*PAT_API_ENTITLEMENT, api=True)
    share, allocated = None, {}
    if ent_raw is not None:
        ent = ent_raw.select(
            F.to_date(opt(ent_raw, "Snapshot Date")).alias("snapshot_date"),
            opt(ent_raw, "Environment Id").cast("string").alias("environment_id"),
            opt(ent_raw, "Environment Allocated").cast("double").alias("allocated"),
            opt(ent_raw, "Tenant Entitled").cast("double").alias("tenant_entitled"),
            opt(ent_raw, "Tenant Prepaid Consumed").cast("double").alias("prepaid"),
            opt(ent_raw, "Tenant PAYG Consumed").cast("double").alias("payg"))
        latest = ent.agg(F.max("snapshot_date")).first()[0]
        snap = ent.where(F.col("snapshot_date") == latest).collect()
        allocated = {r["environment_id"]: r["allocated"] for r in snap if r["environment_id"]}
        if snap:
            share = prepaid_share(snap[0]["prepaid"], snap[0]["payg"], snap[0]["tenant_entitled"])
            print(f"  entitlement snapshot {latest}: prepaid share {share:.0%}, {len(allocated)} environment(s)")
    if share is None:
        share = 1.0
        print("  no entitlement snapshot - treating every credit as prepaid")
    _entitlement_cache.update(share=share, allocated=allocated)
    return share, allocated


RUN_MONTH = datetime.now(timezone.utc).date().replace(day=1)
print(f"run month: {RUN_MONTH:%Y-%m} (UTC)")
if globals().get("SNAPSHOT_MONTH"):
    raise ValueError("SNAPSHOT_MONTH is no longer read. Remove it from the settings cell, or from the "
                     "parameters passed to this notebook: a pipeline Notebook activity's base parameters, "
                     "or the arguments of notebookutils.notebook.run. To load an earlier month's export, "
                     f"put it in {LANDING}/backfill/<YYYY-MM>/. Nothing was written.")


## Tenant daily

The only one of the three with a real date, and the only one Consumption Central plots over time.

The export's days are written in one step, a Delta `replaceWhere`, so a failed write leaves the table
as it was. Each row replaces the stored row with the same day, environment, billing plan and
capacity type, and any licensing API row for its day and environment; other rows on those days
stay. Where two exports share a key, the newer file wins.

In [ ]:
raw = read(*PAT_TENANT, need=[("Usage Date",), ("Environment Id",), ("BillingPlan Id",), ("Capacity Type",)])
if raw is None:
    print("no tenant export - skipping")
else:
    tenant = raw.select(
        F.col(pick(raw, "BillingPlan Id")).cast("string").alias("billing_plan_id"),
        F.col(pick(raw, "BillingPlan Name")).cast("string").alias("billing_plan_name"),
        F.col(pick(raw, "Environment Id")).cast("string").alias("environment_id"),
        F.col(pick(raw, "Environment Name")).cast("string").alias("environment_name"),
        F.col(pick(raw, "Capacity Type")).cast("string").alias("capacity_type"),
        F.col(pick(raw, "Entitled Quantity")).cast("double").alias("entitled_quantity"),
        F.col(pick(raw, "Prepaid Consumed Quantity")).cast("double").alias("prepaid_consumed"),
        F.col(pick(raw, "Pay as you go Consumed Quantity")).cast("double").alias("payg_consumed"),
        # PPAC writes US-style M/d/yyyy with a time component
        F.to_date(F.col(pick(raw, "Usage Date")), "M/d/yyyy H:mm").alias("usage_date"),
        F.col("source_file").alias("source_file"),
        F.col("_order"),
    ).withColumn("_loaded_at", F.current_timestamp())

    bad = tenant.filter(F.col("usage_date").isNull()).count()
    if bad:
        print(f"  ! {bad:,} rows have an unparseable Usage Date - check the format")

    print(f"  {tenant.count():,} rows, "
          f"{tenant.select('environment_id').distinct().count()} environments")
    write_export(tenant, TBL_TENANT, ["usage_date", "environment_id", "billing_plan_id", "capacity_type"],
                 covers=["usage_date", "environment_id"])

## Per agent

No date in the file, so the snapshot month becomes part of the key: the month (UTC) the file was
saved into `Files/landing/studio/`, or its `backfill/<YYYY-MM>/` folder's month. The export's months
are written in one step, as for the tenant export: each row replaces the stored row with the same
key, and the licensing API rows for its month go. Where two exports share a key, the newer file
wins.

In [ ]:
raw = read(*PAT_AGENT, need=[("Agent Id",), ("AI Feature/Billable Feature", "Billable Feature"),
                             ("Channel",), ("Environment Id",)])
if raw is None:
    print("no per-agent export - skipping")
else:
    agent = raw.select(
        F.col("_month").cast("date").alias("snapshot_month"),
        F.col(pick(raw, "Agent Name")).cast("string").alias("agent_name"),
        F.col(pick(raw, "Agent Id")).cast("string").alias("agent_id"),
        F.col(pick(raw, "Product")).cast("string").alias("product"),
        F.col(pick(raw, "AI Feature/Billable Feature", "Billable Feature"))
            .cast("string").alias("billable_feature"),
        F.col(pick(raw, "Billed credit")).cast("double").alias("billed_credit"),
        F.col(pick(raw, "Non-billed credit")).cast("double").alias("non_billed_credit"),
        F.col(pick(raw, "Channel")).cast("string").alias("channel"),
        F.col(pick(raw, "Knowledge Sources")).cast("string").alias("knowledge_sources"),
        F.col(pick(raw, "Tool Used")).cast("string").alias("tool_used"),
        F.col(pick(raw, "LLM Model")).cast("string").alias("llm_model"),
        F.col(pick(raw, "Scenario Name")).cast("string").alias("scenario_name"),
        F.col(pick(raw, "Environment Id")).cast("string").alias("environment_id"),
        F.col(pick(raw, "Environment Name")).cast("string").alias("environment_name"),
        F.col("source_file").alias("source_file"),
        F.col("_order"),
    ).withColumn("_loaded_at", F.current_timestamp())

    print(f"  {agent.count():,} rows, "
          f"{agent.select('agent_id').distinct().count()} agents")
    months = sorted(r[0] for r in agent.select("snapshot_month").distinct().collect())
    print("  snapshot month(s): " + ", ".join(f"{m:%Y-%m}" for m in months))
    # An agent appears once per feature per channel per environment, so all four
    # belong in the key - dropping any of them silently loses rows.
    write_export(agent, TBL_AGENT,
                 ["snapshot_month", "agent_id", "billable_feature", "channel", "environment_id"],
                 covers=["snapshot_month"])

## Per user

Labelled and written like the per-agent export, keyed on the month, user and agent.

In [ ]:
raw = read(*PAT_USER, need=[("User Id",), ("Agent Id",)])
if raw is None:
    print("no per-user export - skipping")
else:
    user = raw.select(
        F.col("_month").cast("date").alias("snapshot_month"),
        F.col(pick(raw, "User Id")).cast("string").alias("user_id"),
        F.lower(F.trim(F.col(pick(raw, "User Email")))).alias("user_email"),
        F.col(pick(raw, "Agent Id")).cast("string").alias("agent_id"),
        F.col(pick(raw, "Agent Name")).cast("string").alias("agent_name"),
        F.col(pick(raw, "Billable credit used")).cast("double").alias("billable_credit_used"),
        F.col(pick(raw, "Credits used")).cast("double").alias("credits_used"),
        F.upper(F.trim(F.col(pick(raw, "M365 Copilot Licensed"))))
            .isin("TRUE", "YES", "1").alias("m365_copilot_licensed"),
        F.col("source_file").alias("source_file"),
        F.col("_order"),
    ).withColumn("_loaded_at", F.current_timestamp())

    print(f"  {user.count():,} rows, "
          f"{user.select('user_email').distinct().count()} users")
    months = sorted(r[0] for r in user.select("snapshot_month").distinct().collect())
    print("  snapshot month(s): " + ", ".join(f"{m:%Y-%m}" for m in months))
    write_export(user, TBL_USER, ["snapshot_month", "user_id", "agent_id"], covers=["snapshot_month"])

## From the licensing API

Only with `READ_API_FILES = True`. A Power Automate flow calls the Power Platform licensing API
and drops three files: per-agent and per-user credits by day, and a snapshot of the tenant's
entitlement with each environment's allocation. The flow should keep only the newest of each:
every matching file is read, so two files covering the same day would be added together.

- `studio_agent_daily`: the days in the file replace the same days already loaded. Environment
  names come from the flow, or from the entitlement snapshot when the flow couldn't read them.
- `studio_agent`: each month is the sum of its days, unless an export already covers that month.
- `studio_tenant_daily`: one row per environment per day. The API doesn't split prepaid from
  pay-as-you-go by day, so each day is split by the tenant's month-to-date prepaid share. The
  entitled quantity is the environment's current allocation. An export for the same day and
  environment replaces these rows.

Each table is written in one step: a Delta `replaceWhere` replaces the file's days (or months) in a
single commit. A write that fails leaves its table as it was, so no day is lost: those days keep
their earlier figures until a later run's file covers them again. Days and months the file doesn't
have keep their rows, and a month in `studio_agent` is summed again on every run whose file has a
day in it.

In [ ]:
api_raw = read(*PAT_API_AGENT, api=True)
if api_raw is None:
    print("no licensing API files - skipping")
else:
    keys = ["usage_date", "agent_id", "billable_feature", "channel", "environment_id"]
    rows = api_raw.select(
        F.to_date(opt(api_raw, "Usage Date")).alias("usage_date"),
        opt(api_raw, "Agent Id").cast("string").alias("agent_id"),
        opt(api_raw, "Agent Name").cast("string").alias("agent_name"),
        opt(api_raw, "Environment Id").cast("string").alias("environment_id"),
        opt(api_raw, "Environment Name").cast("string").alias("environment_name"),
        opt(api_raw, "Feature").cast("string").alias("billable_feature"),
        opt(api_raw, "Channel").cast("string").alias("channel"),
        opt(api_raw, "LLM Model").cast("string").alias("llm_model"),
        opt(api_raw, "Tool Used").cast("string").alias("tool_used"),
        opt(api_raw, "Knowledge Sources").cast("string").alias("knowledge_sources"),
        opt(api_raw, "Billed credit").cast("double").alias("billed_credit"),
        opt(api_raw, "Non-billed credit").cast("double").alias("non_billed_credit"),
        opt(api_raw, "Users").cast("double").alias("users"),
    ).where(F.col("usage_date").isNotNull() & F.col("agent_id").isNotNull())

    # The flow fills environment names from the environments API; the entitlement snapshot fills any it missed.
    names_raw = read(*PAT_API_ENTITLEMENT, api=True)
    if names_raw is not None:
        names = (names_raw.select(opt(names_raw, "Environment Id").cast("string").alias("environment_id"),
                                  opt(names_raw, "Environment Name").cast("string").alias("env_name"))
                 .where("env_name IS NOT NULL AND env_name <> ''")
                 .dropDuplicates(["environment_id"]))
        rows = (rows.join(names, "environment_id", "left")
                .withColumn("environment_name", F.coalesce(F.when(F.col("environment_name") != "", F.col("environment_name")), F.col("env_name")))
                .drop("env_name"))

    def joined(c):
        return F.array_join(F.array_sort(F.collect_set(c)), ", ").alias(c)

    daily = (rows.groupBy(*keys).agg(
                F.first("agent_name", True).alias("agent_name"),
                F.first("environment_name", True).alias("environment_name"),
                F.sum("billed_credit").alias("billed_credit"),
                F.sum("non_billed_credit").alias("non_billed_credit"),
                F.max("users").alias("users"),
                joined("llm_model"), joined("tool_used"), joined("knowledge_sources"))
             .withColumn("source_file", F.lit(API_SOURCE))
             .withColumn("_loaded_at", F.current_timestamp()))
    days = [r[0] for r in daily.select("usage_date").distinct().collect()]
    print(f"  {daily.count():,} agent-days over {len(days)} day(s)")

    if days:
        replace_rows(daily, TBL_AGENT_DAILY, date_in("usage_date", days))

    # studio_agent: months no export covers.
    months = sorted({d.replace(day=1) for d in days})
    covered = set()
    if spark.catalog.tableExists(TBL_AGENT):
        covered = {r[0] for r in spark.table(TBL_AGENT)
                   .where(f"source_file IS NULL OR source_file <> '{API_SOURCE}'")
                   .select("snapshot_month").distinct().collect()}
    api_months = [m for m in months if m not in covered]
    for m in months:
        if m in covered:
            print(f"  {m:%Y-%m}: an export covers this month, so the API figures aren't used")
    if api_months:
        monthly = (spark.table(TBL_AGENT_DAILY)
                   .withColumn("snapshot_month", F.trunc("usage_date", "month"))
                   .where(F.col("snapshot_month").isin(api_months))
                   .groupBy("snapshot_month", "agent_id", "billable_feature", "channel", "environment_id")
                   .agg(F.first("agent_name", True).alias("agent_name"),
                        F.first("environment_name", True).alias("environment_name"),
                        F.sum("billed_credit").alias("billed_credit"),
                        F.sum("non_billed_credit").alias("non_billed_credit"),
                        F.first("llm_model", True).alias("llm_model"),
                        F.first("tool_used", True).alias("tool_used"),
                        F.first("knowledge_sources", True).alias("knowledge_sources"))
                   .select("snapshot_month", "agent_name", "agent_id",
                           F.lit("Copilot Studio").alias("product"), "billable_feature",
                           "billed_credit", "non_billed_credit", "channel", "knowledge_sources",
                           "tool_used", "llm_model", F.lit(None).cast("string").alias("scenario_name"),
                           "environment_id", "environment_name",
                           F.lit(API_SOURCE).alias("source_file"))
                   .withColumn("_loaded_at", F.current_timestamp()))
        replace_rows(monthly, TBL_AGENT, f"{date_in('snapshot_month', api_months)} AND source_file = '{API_SOURCE}'")

    # studio_tenant_daily: the entitlement snapshot gives the prepaid share and each environment's allocation.
    share, allocated = entitlement()

    alloc = spark.createDataFrame([(k, v) for k, v in allocated.items()] or [("", None)],
                                  "environment_id string, entitled_quantity double")
    tenant_api = (daily.groupBy("usage_date", "environment_id")
                  .agg(F.first("environment_name", True).alias("environment_name"),
                       F.sum("billed_credit").alias("consumed"))
                  .join(alloc, "environment_id", "left")
                  .select(F.lit(API_BILLING_PLAN).alias("billing_plan_id"),
                          F.lit("Power Platform licensing API").alias("billing_plan_name"),
                          "environment_id", "environment_name",
                          F.lit("MCSMessages").alias("capacity_type"),
                          F.coalesce("entitled_quantity", F.lit(0.0)).alias("entitled_quantity"),
                          (F.col("consumed") * F.lit(share)).alias("prepaid_consumed"),
                          (F.col("consumed") * F.lit(1.0 - share)).alias("payg_consumed"),
                          "usage_date", F.lit(API_SOURCE).alias("source_file"))
                  .withColumn("_loaded_at", F.current_timestamp()))
    if spark.catalog.tableExists(TBL_TENANT):
        exported = (spark.table(TBL_TENANT)
                    .where(~F.coalesce(F.col("source_file").isin(*API_SOURCES), F.lit(False)))
                    .select("usage_date", "environment_id").distinct())
        # Null-safe, so an export row with no environment ID covers the API row with none.
        tenant_api = (tenant_api.alias("t")
                      .join(exported.alias("s"), F.expr(same(["usage_date", "environment_id"])), "left_anti"))
    if days:
        # One write replaces the days' API rows, and any 'ppac-api-users' row a new row shares a key
        # with, as a merge would. The other rows `where` selects are written back as they were, among
        # them any export row with no environment ID: `exported` leaves no new row on its key.
        tkeys = ["usage_date", "environment_id", "billing_plan_id", "capacity_type"]
        where = (f"{date_in('usage_date', days)} AND (source_file IN ('{API_SOURCE}', '{API_USER_SOURCE}')"
                 " OR environment_id IS NULL)")
        kept = None
        if spark.catalog.tableExists(TBL_TENANT):
            match = " AND ".join(f"t.{k} <=> s.{k}" for k in tkeys)
            kept = (spark.table(TBL_TENANT).where(where).where(f"NOT (source_file <=> '{API_SOURCE}')")
                    .alias("t").join(tenant_api.alias("s"), F.expr(match), "left_anti"))
        replace_rows(tenant_api, TBL_TENANT, where, keep=kept)

## Per user, from the licensing API

The per-user figures are best effort: the flow reads them from a licensing API route that isn't
documented, and skips them if it fails. They name each user by Entra object ID, so this looks up
the UPN in Microsoft Graph (`/users` with `id in (...)`, batched). A failed lookup never fails the
run: those users show by ID.

- `studio_user_daily`: one row per day, user, environment and agent. The days in the file replace the
  same days already loaded, in one step as above.
- `studio_user`: each month is the sum of its days, unless an export already covers that month.

In [ ]:
import time
import requests
from urllib.parse import quote

_GUID = re.compile(r"^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$")
_IDS_PER_QUERY = 15
_QUERIES_PER_BATCH = 20


def _graph_token():
    resp = requests.post(
        f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token",
        data={"grant_type": "client_credentials", "client_id": CLIENT_ID,
              "client_secret": CLIENT_SECRET, "scope": "https://graph.microsoft.com/.default"},
        timeout=60)
    resp.raise_for_status()
    return resp.json()["access_token"]


def _no_access(status):
    return PermissionError(f"Graph returned HTTP {status}. Grant the app registration "
                           "Microsoft Graph User.Read.All (application) and admin consent.")


def resolve_upns(ids):
    """{object id (lower case): UPN (lower case)} for the IDs Graph knows."""
    headers = {"Authorization": f"Bearer {_graph_token()}", "Content-Type": "application/json"}
    found = {}
    queries = [ids[i:i + _IDS_PER_QUERY] for i in range(0, len(ids), _IDS_PER_QUERY)]
    for start in range(0, len(queries), _QUERIES_PER_BATCH):
        pending = {str(n): q for n, q in enumerate(queries[start:start + _QUERIES_PER_BATCH])}
        for attempt in range(6):
            if not pending:
                break
            body = {"requests": [
                {"id": key, "method": "GET",
                 "url": "/users?$filter=" + quote("id in (" + ",".join(f"'{i}'" for i in q) + ")")
                        + "&$select=id,userPrincipalName"}
                for key, q in pending.items()]}
            r = requests.post("https://graph.microsoft.com/v1.0/$batch", headers=headers, json=body, timeout=120)
            if r.status_code in (401, 403):
                raise _no_access(r.status_code)
            if r.status_code == 429 or r.status_code >= 500:
                time.sleep(min(float(r.headers.get("Retry-After", 2 ** attempt)), 60))
                continue
            r.raise_for_status()
            wait = 0.0
            for item in r.json().get("responses", []):
                status = int(item.get("status") or 0)
                if status == 200:
                    for u in (item.get("body") or {}).get("value", []):
                        if u.get("id") and u.get("userPrincipalName"):
                            found[u["id"].lower()] = u["userPrincipalName"].strip().lower()
                    pending.pop(item.get("id"), None)
                elif status in (401, 403):
                    raise _no_access(status)
                elif status == 429 or status >= 500:
                    wait = max(wait, float((item.get("headers") or {}).get("Retry-After", 2 ** attempt)))
                else:
                    print(f"  UPN lookup query skipped: HTTP {status}")
                    pending.pop(item.get("id"), None)
            if pending:
                time.sleep(min(wait or 2 ** attempt, 60))
        if pending:
            print(f"  {len(pending)} UPN lookup query(ies) still throttled; those users show by ID.")
    return found


def blank_null(c):
    return F.when(F.trim(c) != "", F.trim(c))


user_raw = read(*PAT_API_USER, api=True)
if user_raw is None:
    print("no licensing API user files - skipping")
else:
    ukeys = ["usage_date", "user_id", "environment_id", "agent_id"]
    udaily = (user_raw.select(
                F.to_date(opt(user_raw, "Usage Date")).alias("usage_date"),
                F.lower(blank_null(opt(user_raw, "User Id").cast("string"))).alias("user_id"),
                blank_null(opt(user_raw, "Environment Id").cast("string")).alias("environment_id"),
                blank_null(opt(user_raw, "Agent Id").cast("string")).alias("agent_id"),
                opt(user_raw, "Billed credit").cast("double").alias("billed_credit"),
                opt(user_raw, "Non-billed credit").cast("double").alias("non_billed_credit"))
              .where(F.col("usage_date").isNotNull() & F.col("user_id").isNotNull())
              .groupBy(*ukeys)
              .agg(F.sum("billed_credit").alias("billed_credit"),
                   F.sum("non_billed_credit").alias("non_billed_credit")))

    # UPNs already known from earlier loads and exports, then Graph for the rest.
    known = {}
    if spark.catalog.tableExists(TBL_USER):
        known.update({r[0].lower(): r[1] for r in spark.table(TBL_USER)
                      .where("user_id IS NOT NULL AND user_email IS NOT NULL")
                      .select("user_id", "user_email").distinct().collect()})
    if spark.catalog.tableExists(TBL_USER_DAILY):
        known.update({r[0]: r[1] for r in spark.table(TBL_USER_DAILY)
                      .where("user_upn IS NOT NULL")
                      .select("user_id", "user_upn").distinct().collect()})
    ids = sorted(i for i in {r[0] for r in udaily.select("user_id").distinct().collect()} - set(known)
                 if _GUID.match(i))
    if not ids:
        pass
    elif not RESOLVE_UPNS:
        print("  RESOLVE_UPNS is off, so users show by ID")
    elif any(str(v).startswith("<") for v in (TENANT_ID, CLIENT_ID, CLIENT_SECRET)):
        print("  credentials are not set, so users show by ID")
    else:
        try:
            found = resolve_upns(ids)
            known.update(found)
            print(f"  resolved {len(found):,} of {len(ids):,} user object IDs to UPNs")
        except Exception as e:
            print(f"  UPN lookup skipped: {e}")

    upns = spark.createDataFrame(list(known.items()) or [("", None)], "user_id string, user_upn string")
    udaily = (udaily.join(upns, "user_id", "left")
              .select(*ukeys, "user_upn", "billed_credit", "non_billed_credit")
              .withColumn("source_file", F.lit(API_SOURCE))
              .withColumn("_loaded_at", F.current_timestamp()))
    udays = [r[0] for r in udaily.select("usage_date").distinct().collect()]
    print(f"  {udaily.count():,} user rows over {len(udays)} day(s)")
    if udays:
        replace_rows(udaily, TBL_USER_DAILY, date_in("usage_date", udays))

    # studio_user: months no export covers.
    umonths = sorted({d.replace(day=1) for d in udays})
    ucovered = set()
    if spark.catalog.tableExists(TBL_USER):
        ucovered = {r[0] for r in spark.table(TBL_USER)
                    .where(f"source_file IS NULL OR source_file <> '{API_SOURCE}'")
                    .select("snapshot_month").distinct().collect()}
    user_months = [m for m in umonths if m not in ucovered]
    for m in umonths:
        if m in ucovered:
            print(f"  {m:%Y-%m}: an export covers this month, so the API user figures aren't used")
    if user_months:
        agent_names = (spark.table(TBL_AGENT_DAILY).groupBy("agent_id")
                       .agg(F.first("agent_name", True).alias("agent_name"))
                       if spark.catalog.tableExists(TBL_AGENT_DAILY)
                       else spark.createDataFrame([], "agent_id string, agent_name string"))
        umonthly = (spark.table(TBL_USER_DAILY)
                    .withColumn("snapshot_month", F.trunc("usage_date", "month"))
                    .where(F.col("snapshot_month").isin(user_months))
                    .groupBy("snapshot_month", "user_id", "agent_id")
                    .agg(F.first("user_upn", True).alias("user_email"),
                         F.sum("billed_credit").alias("billable_credit_used"),
                         F.sum(F.coalesce("billed_credit", F.lit(0.0))
                               + F.coalesce("non_billed_credit", F.lit(0.0))).alias("credits_used"))
                    .join(agent_names, "agent_id", "left")
                    .select("snapshot_month", "user_id", "user_email", "agent_id", "agent_name",
                            "billable_credit_used", "credits_used",
                            F.lit(None).cast("boolean").alias("m365_copilot_licensed"),
                            F.lit(API_SOURCE).alias("source_file"))
                    .withColumn("_loaded_at", F.current_timestamp()))
        replace_rows(umonthly, TBL_USER, f"{date_in('snapshot_month', user_months)} AND source_file = '{API_SOURCE}'")

## Tenant daily from the per-user file

The flow's per-agent file is sometimes missing or empty while the per-user file has credits. The
tenant totals, the daily chart and the budget runway all read `studio_tenant_daily`, so they would stay
blank. For each day and environment that no export or agent file covers, this sums
`studio_user_daily` into one row with `source_file = 'ppac-api-users'`:

- `prepaid_consumed`: billed credits times the prepaid share, plus the non-billed credits (those are
  absorbed by the entitlement, so never pay-as-you-go).
- `payg_consumed`: billed credits times the pay-as-you-go share.
- `entitled_quantity`: the environment's current allocation, as for the agent file.

The rows are rebuilt on every run and replaced in one step, so a failed write keeps the last run's
rows, and a later export or agent file for the same day replaces them.
Per-user figures leave out credits no user is tied to (autonomous agents, for example), so these
totals can be a little lower than an export would show.

In [ ]:
# Rebuilt on every run: the write at the end replaces every 'ppac-api-users' row in one commit.
tenant_users = None

if not READ_API_FILES:
    print("licensing API files are off - nothing to sum")
elif not spark.catalog.tableExists(TBL_USER_DAILY):
    print("no per-user daily table - nothing to sum")
else:
    by_day = (spark.table(TBL_USER_DAILY)
              .where(F.col("usage_date").isNotNull())
              .groupBy("usage_date", "environment_id")
              .agg(F.sum(F.coalesce("billed_credit", F.lit(0.0))).alias("billed"),
                   F.sum(F.coalesce("non_billed_credit", F.lit(0.0))).alias("non_billed")))
    if spark.catalog.tableExists(TBL_TENANT):
        # The last run's per-user rows are still stored. The write below replaces them, so they don't count.
        covered = (spark.table(TBL_TENANT).where(f"NOT (source_file <=> '{API_USER_SOURCE}')")
                   .select("usage_date", "environment_id").distinct())
        by_day = by_day.join(covered, (by_day.usage_date == covered.usage_date)
                             & by_day.environment_id.eqNullSafe(covered.environment_id), "left_anti")
    if by_day.limit(1).count() == 0:
        print("  every day with user credits already has tenant rows")
    else:
        share, allocated = entitlement()
        alloc = spark.createDataFrame([(k, v) for k, v in allocated.items()] or [("", None)],
                                      "environment_id string, entitled_quantity double")
        names = (spark.table(TBL_AGENT_DAILY)
                 .where("environment_name IS NOT NULL AND environment_name <> ''")
                 .groupBy("environment_id").agg(F.first("environment_name", True).alias("environment_name"))
                 if spark.catalog.tableExists(TBL_AGENT_DAILY)
                 else spark.createDataFrame([], "environment_id string, environment_name string"))
        tenant_users = (by_day.join(alloc, "environment_id", "left")
                        .join(names, "environment_id", "left")
                        .select(F.lit(API_BILLING_PLAN).alias("billing_plan_id"),
                                F.lit("Power Platform licensing API (per user)").alias("billing_plan_name"),
                                "environment_id", "environment_name",
                                F.lit("MCSMessages").alias("capacity_type"),
                                F.coalesce("entitled_quantity", F.lit(0.0)).alias("entitled_quantity"),
                                (F.col("billed") * F.lit(share) + F.col("non_billed")).alias("prepaid_consumed"),
                                (F.col("billed") * F.lit(1.0 - share)).alias("payg_consumed"),
                                "usage_date", F.lit(API_USER_SOURCE).alias("source_file"))
                        .withColumn("_loaded_at", F.current_timestamp()))
        print(f"  {tenant_users.count():,} tenant day(s) summed from the per-user file")

if tenant_users is not None:
    replace_rows(tenant_users, TBL_TENANT, f"source_file = '{API_USER_SOURCE}'")
elif READ_API_FILES and spark.catalog.tableExists(TBL_TENANT):
    DeltaTable.forName(spark, TBL_TENANT).delete(F.col("source_file") == API_USER_SOURCE)

## Check

The tenant file is the source of truth for totals. The per-agent and per-user files are
attributions of it, and they will not add up to it — agent and environment consumption carries no
user, so the per-user total is always the smaller number. That gap is expected and Consumption Central says
so on the page rather than hiding it.

In [ ]:
for t in (TBL_TENANT, TBL_AGENT, TBL_USER, TBL_AGENT_DAILY, TBL_USER_DAILY):
    if spark.catalog.tableExists(t):
        n = spark.table(t).count()
        print(f"{t:24s} {n:>8,} rows")
    else:
        print(f"{t:24s}        - not created")

if spark.catalog.tableExists(TBL_TENANT):
    spark.sql(f"""
        SELECT  CASE WHEN source_file IN ('{API_SOURCE}', '{API_USER_SOURCE}') THEN source_file
                     ELSE 'export' END AS source,
                MIN(usage_date) AS earliest, MAX(usage_date) AS latest,
                COUNT(DISTINCT usage_date) AS days,
                ROUND(SUM(prepaid_consumed)) AS prepaid,
                ROUND(SUM(payg_consumed))    AS payg
        FROM    {TBL_TENANT}
        GROUP BY 1
    """).show(truncate=False)